# V22 E016 full 46-event availability shadow

Evaluation only. This notebook does not train, assign links, mutate graphs, or infer edges. It runs the frozen epoch-24 clean checkpoint on all 46 registered development events.

In [ ]:
from pathlib import Path
import glob, hashlib, json, os, subprocess, sys, shutil
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
ROOT = Path('/kaggle/working/Atabey')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'v22-upstream-division-availability', 'https://github.com/drosadocastro-bit/Atabey.git', str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
train_candidates = [Path(p) for p in glob.glob('/kaggle/input/**/train', recursive=True)]
train_candidates = [p for p in train_candidates if len(list(p.glob('*.zarr'))) >= 190]
assert train_candidates, 'Competition train directory not found'
TRAIN_DIR = train_candidates[0]
support_candidates = [Path(p) for p in glob.glob('/kaggle/input/**/repo', recursive=True) if (Path(p) / 'scripts/predict_unet_transformer.py').exists()]
assert support_candidates, 'Attach the public E016 support-pack dataset'
SUPPORT_REPO = support_candidates[0]
weight_candidates = [Path(p) for p in glob.glob('/kaggle/input/**/*.pth', recursive=True) if Path(p).name == 'edge_predictor_best.pth']
assert weight_candidates, 'Attach the downloaded v22_e016_clean_checkpoint.zip as a Kaggle dataset'
WEIGHTS = weight_candidates[0]
EXPECTED_HASH = '02e1d65756c3dc5928f68a66a8b0ef99be2a6905fa7bc017aa1d87dbe632fd03'
actual_hash = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
assert actual_hash == EXPECTED_HASH, (actual_hash, EXPECTED_HASH)
print('Train:', TRAIN_DIR)
print('Support:', SUPPORT_REPO)
print('Checkpoint:', WEIGHTS)
print('Checkpoint hash verified:', actual_hash)

In [ ]:
# Create a temporary 46-event detector fixture with the frozen checkpoint hash.
fixture = json.loads((ROOT / 'tests/fixtures/v22_unet_detection_development_46.json').read_text())
fixture['name'] = 'v22_e016_epoch24_full_46_shadow_v1'
fixture['expected_weight_sha256'] = EXPECTED_HASH
fixture['pool_kernel_um'] = 5.0
FIXTURE = Path('/kaggle/working/v22_e016_epoch24_full_46_fixture.json')
FIXTURE.write_text(json.dumps(fixture, indent=2))
assert len(fixture['cases']) == 46
print('Fixture locked:', FIXTURE, 'cases:', len(fixture['cases']))

In [ ]:
PEAKS = Path('/kaggle/working/v22_e016_epoch24_full_46_peaks.csv')
DETECTOR_CSV = Path('/kaggle/working/v22_e016_epoch24_full_46_detector.csv')
DETECTOR_SUMMARY = Path('/kaggle/working/v22_e016_epoch24_full_46_detector_summary.json')
subprocess.run([sys.executable, str(ROOT / 'scripts/run_v22_unet_detection_shadow.py'), '--train-dir', str(TRAIN_DIR), '--support-repo', str(SUPPORT_REPO), '--weights', str(WEIGHTS), '--fixture', str(FIXTURE), '--output-csv', str(DETECTOR_CSV), '--output-summary', str(DETECTOR_SUMMARY), '--output-peaks', str(PEAKS)], check=True)
print('Detector shadow complete:', DETECTOR_CSV, PEAKS)

In [ ]:
AVAIL_CSV = Path('/kaggle/working/v22_e016_epoch24_full_46_availability.csv')
AVAIL_SUMMARY = Path('/kaggle/working/v22_e016_epoch24_full_46_availability_summary.json')
AVAIL_REPORT = Path('/kaggle/working/V22_E016_FULL_46_AVAILABILITY_RESULTS.md')
subprocess.run([sys.executable, str(ROOT / 'scripts/run_v22_unet_official_action_availability.py'), '--train-dir', str(TRAIN_DIR), '--peaks', str(PEAKS), '--output', str(AVAIL_CSV), '--summary', str(AVAIL_SUMMARY), '--report', str(AVAIL_REPORT), '--resume'], check=True)
print(AVAIL_SUMMARY.read_text())
print('Report:', AVAIL_REPORT)

In [ ]:
# Bundle compact outputs for download; no competition volumes are copied.
bundle = Path('/kaggle/working/v22_e016_full_46_availability_outputs')
bundle.mkdir(exist_ok=True)
for path in [PEAKS, DETECTOR_CSV, DETECTOR_SUMMARY, AVAIL_CSV, AVAIL_SUMMARY, AVAIL_REPORT]:
    shutil.copy2(path, bundle / path.name)
shutil.make_archive('/kaggle/working/v22_e016_full_46_availability_outputs', 'zip', bundle)
print('Download v22_e016_full_46_availability_outputs.zip')